In [1]:
import pandas as pd
import string
import re
import nltk
import time


stopword = nltk.corpus.stopwords.words('english')
ps = nltk.PorterStemmer()

df = pd.read_csv('data/train.csv')

# Kagel data is weird...
test = pd.read_csv('data/test.csv')
tlabels= pd.read_csv('data/test_labels.csv')

valid_tlabels = tlabels[tlabels['toxic'] != -1]
valid_test_df = test[test['id'].isin(valid_tlabels['id'])]

vtest = pd.merge(valid_test_df, valid_tlabels, on='id')

df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [2]:
from scipy.sparse import hstack
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

def preprocess_text(series):
     # @somone to @
    lst = []
    lstn = []
    length = len(series)
    for i, text in enumerate(series):
        # mark @, # and links
        text = ' '.join(['_at_someone_' if '@' in word else word for word in text.split()])
        text = ' '.join(['_link_' if 'http' in word or 'www.' in word else word for word in text.split()])
        text = ' '.join(['_hashtag_' if '#' in word else word for word in text.split()])
        # remove all other punctuation
        text = ''.join([word.lower() for word in text if word not in string.punctuation.replace('@','').replace('#','').replace('_','')])

        # the real work, pos tagging to reduce Nouns and other things that don't matter
        tokens = []
        nouns = []
        for a in nltk.pos_tag(text.split()):
            # keep these as-is for reasons
            if '_link_' in a[0] or '_at_someone_' in a[0] or '_hashtag_' in a[0]:
                tokens.append(a[0])
                nouns.append(a[0])
            # lemmatize these to reduce parts of speach issues
            elif 'NN' in a[1] or 'JJ' in a[1] or 'RB' in a[1] or 'V' in a[1]:
                tokens.append(a[1])
                try:
                    nouns.append(nltk.PorterStemmer().stem(a[0]))
                except:
                    print('Recursion error',a)
            else:
                tokens.append(a[1])
                #nouns.append(nltk.PorterStemmer().stem(a[0]))
        lst.append(' '.join(tokens))
        lstn.append(' '.join(nouns))
        if i%100 == 0:
            print("\r{:.2f}% Complete".format(100*((i+1)/length), a), end="", flush=True)
    print("\r100.00% Complete".format(100*((i+1)/length)), end="", flush=True)
    return lst, lstn

def vectorize_data(df, nounVectorizer, gramVectorizer, charVectorizer, scaler):
    gV = gramVectorizer.transform(df['pos'])
    nV = nounVectorizer.transform(df['stems'])
    nC = charVectorizer.transform(df['comment_text'])
    transformed_data = hstack([gV, nV, nC])

    if scaler != None:
        scaler.fit(transformed_data)
    return transformed_data

def fit_vectorizers(df):
    # gram is based on the work I diud in howRussian
    gramVectorizer =  TfidfVectorizer(ngram_range=(2,5)
                                        , min_df = 10
                                        , max_df = .7
                                        , stop_words = None
                                        , lowercase=False              # Protect POS tag case sensitivity
                                        , token_pattern=r'\S+'          # Tokenize strictly by space-separated tags/tokens
                                       )
    gramVectorizer.fit(df['pos'])
    
    
    
    # a simple vectorizer on the nouns to include specific words because reasons, ideal at word and bigram
    nounVectorizer =  TfidfVectorizer(ngram_range=(1,2)
                                        , min_df = 5
                                        , max_df = 0.75
                                        , stop_words = nltk.corpus.stopwords.words('english')
                                       )
    nounVectorizer.fit(df['stems'])


    # char_vec to catch weird mispellings...
    charVectorizer = TfidfVectorizer(ngram_range=(4, 5)
                                        , analyzer='char'
                                        , sublinear_tf=True
                                        , min_df=5
                                        , max_df = 0.5
                                        , max_features=50000
                                       )
    charVectorizer.fit(df['comment_text'])

    
    gV = gramVectorizer.transform(df['pos'])
    nV = nounVectorizer.transform(df['stems'])
    nC = charVectorizer.transform(df['comment_text'])
    transformed_data = hstack([gV, nV, nC])

    scaler = StandardScaler(with_mean=False)
    # scaler = None
    scaler.fit(transformed_data)
    return gramVectorizer, nounVectorizer, charVectorizer, scaler

In [3]:
df['pos'], df['stems'] = preprocess_text(df['comment_text'])
df.head()

72.44% CompleteRecursion error ('yyyyyyyyyyyyyyybvgtfrrrrrbyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyy

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,pos,stems
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,NN WRB DT NNS VBN IN PRP$ JJ NN NN NN VBD VBN ...,explan whi edit made usernam hardcor metallica...
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,NN PRP VBZ DT NN VBZ JJ RB VBN IN NNS NN CD NN...,daww match background colour im seemingli stuc...
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,NN NN VBP RB RB VBG TO VB NN PRP$ RB IN DT NN ...,hey man im realli not tri edit war just guy is...
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,RBR JJ JJ VBP DT JJ NNS IN NN NN VBD IN DT NN ...,more i cant make real suggest improv i wonder ...
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,PRP VBP VBP PRP$ NN DT NN PRP VBP WP NN NNS IN,sir are hero chanc rememb page that


In [4]:
vtest['pos'], vtest['stems'] = preprocess_text(vtest['comment_text'])
vtest.head()

100.00% Complete

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,pos,stems
0,0001ea8717f6de06,Thank you for understanding. I think very high...,0,0,0,0,0,0,NN PRP IN VBG JJ VBP RB RB IN PRP CC MD RB VB ...,thank understand i think veri highli not rever...
1,000247e83dcc1211,:Dear god this site is horrible.,0,0,0,0,0,0,JJ NN DT NN VBZ JJ,dear god site is horribl
2,0002f87b16116a7f,"""::: Somebody will invariably try to add Relig...",0,0,0,0,0,0,NN MD RB VB TO VB NN RB PRP VBP DT NN NNS VBP ...,somebodi invari tri add religion realli mean w...
3,0003e1cccfd5a40a,""" \n\n It says it right there that it IS a typ...",0,0,0,0,0,0,PRP VBZ PRP RB EX IN PRP VBZ DT NN DT NN IN NN...,say right is type type institut is need case a...
4,00059ace3e3e9a53,""" \n\n == Before adding a new product to the l...",0,0,0,0,0,0,IN VBG DT JJ NN TO DT NN VBP JJ PRP$ NN IN VBG...,ad new product list make sure relev ad new pro...


In [5]:
gramVectorizer, nounVectorizer, charVectorizer, scaler = fit_vectorizers(df)
X = vectorize_data(df, nounVectorizer, gramVectorizer, charVectorizer, scaler)

print('X shape', X.shape)

Xt = vectorize_data(vtest, nounVectorizer, gramVectorizer, charVectorizer, scaler)
print('Xt shape', Xt.shape)

X shape (159571, 387361)
Xt shape (63978, 387361)


In [6]:
df['flag'] = df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].max(axis=1)
vtest['flag'] = vtest[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].max(axis=1)
vtest.head()


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,pos,stems,flag
0,0001ea8717f6de06,Thank you for understanding. I think very high...,0,0,0,0,0,0,NN PRP IN VBG JJ VBP RB RB IN PRP CC MD RB VB ...,thank understand i think veri highli not rever...,0
1,000247e83dcc1211,:Dear god this site is horrible.,0,0,0,0,0,0,JJ NN DT NN VBZ JJ,dear god site is horribl,0
2,0002f87b16116a7f,"""::: Somebody will invariably try to add Relig...",0,0,0,0,0,0,NN MD RB VB TO VB NN RB PRP VBP DT NN NNS VBP ...,somebodi invari tri add religion realli mean w...,0
3,0003e1cccfd5a40a,""" \n\n It says it right there that it IS a typ...",0,0,0,0,0,0,PRP VBZ PRP RB EX IN PRP VBZ DT NN DT NN IN NN...,say right is type type institut is need case a...,0
4,00059ace3e3e9a53,""" \n\n == Before adding a new product to the l...",0,0,0,0,0,0,IN VBG DT JJ NN TO DT NN VBP JJ PRP$ NN IN VBG...,ad new product list make sure relev ad new pro...,0


In [7]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier



col = 'flag'

y = df[col]
yt = vtest[col]


gatekeeper = LogisticRegression(
    C = 0.5
    , penalty='l2'
    , solver='sag'
    , class_weight='balanced'
    , n_jobs = -1
)




gatekeeper.fit(X,y)
print('Training Data {:.3f}, Test Data Score: {:.3f}'.format(gatekeeper.score(X,y),gatekeeper.score(Xt,yt)))

y_pred_probs = gatekeeper.predict_proba(X)[:, 1]
yt_pred_probs = gatekeeper.predict_proba(Xt)[:, 1]

# Calculate ROC-AUC for the single binary label
auc_score = roc_auc_score(y, y_pred_probs)

auc_scoret = roc_auc_score(yt, yt_pred_probs)

print(f"TRAINING ROC-AUC Score: {auc_score:.4f}, Test ROC-AUC Score: {auc_scoret:.4f}")

df['pred'] = gatekeeper.predict_proba(X)[:, 1]
df.head()

C:\Users\darin\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Training Data 0.974, Test Data Score: 0.888
TRAINING ROC-AUC Score: 0.9970, Test ROC-AUC Score: 0.9632


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,pos,stems,flag,pred
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,NN WRB DT NNS VBN IN PRP$ JJ NN NN NN VBD VBN ...,explan whi edit made usernam hardcor metallica...,0,0.030489
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,NN PRP VBZ DT NN VBZ JJ RB VBN IN NNS NN CD NN...,daww match background colour im seemingli stuc...,0,0.019345
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,NN NN VBP RB RB VBG TO VB NN PRP$ RB IN DT NN ...,hey man im realli not tri edit war just guy is...,0,0.073244
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,RBR JJ JJ VBP DT JJ NNS IN NN NN VBD IN DT NN ...,more i cant make real suggest improv i wonder ...,0,0.002369
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,PRP VBP VBP PRP$ NN DT NN PRP VBP WP NN NNS IN,sir are hero chanc rememb page that,0,0.160274


In [8]:
df1 = df[df['flag']==1]
df2 = df[df['flag']==0]
df2 = df2.sort_values('pred', ascending=False)
df2 = df2.head(df1.shape[0])
train = pd.concat([df1,df2]).sort_index()
Xtrain = X[train.index]
train.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,pos,stems,flag,pred
6,0002bcb3da6cb337,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1,1,1,0,1,0,NN IN PRP VBP RB IN PRP$ NN,cocksuck piss around work,1,0.986965
8,00037261f536c51d,Sorry if the word 'nonsense' was offensive to ...,0,0,0,0,0,0,NN IN DT NN NN VBD JJ TO PRP RB VBZ RB VBG TO ...,sorri word nonsens wa offens anyway im not int...,0,0.250234
12,0005c987bdfc9d4b,Hey... what is it..\n@ | talk .\nWhat is it......,1,0,0,0,0,0,NN WP VBZ PRP _at_someone_ VB WP VBZ PRP DT JJ...,hey is _at_someone_ talk is exclus group wp ta...,1,0.719698
16,0007e25b2121310b,"Bye! \n\nDon't look, come or think of comming ...",1,0,0,0,0,0,NN NN VBP VBP CC VBP IN VBG RP NN,bye dont look come think com tosser,1,0.701621
25,000eefc67a2c930f,Radial symmetry \n\nSeveral now extinct lineag...,0,0,0,0,0,0,JJ NN JJ RB JJ NNS VBN IN DT NN VBD JJ JJ IN N...,radial symmetri sever now extinct lineag inclu...,0,0.216981


In [9]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier


target_cols = ['flag']  #,'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
ytrain = X
col = 'flag'

y = train[col]
yt = vtest[col]


gatekeeper = LogisticRegression(
    C = 0.5
    , penalty='l2'
    , solver='sag'
    , class_weight='balanced'
    , n_jobs = -1
)

#model = RandomForestClassifier()


gatekeeper.fit(Xtrain,y)
print('Training Data {:.3f}, Test Data Score: {:.3f}'.format(gatekeeper.score(Xtrain,y),gatekeeper.score(Xt,yt)))

y_pred_probs = gatekeeper.predict_proba(Xtrain)[:, 1]
yt_pred_probs = gatekeeper.predict_proba(Xt)[:, 1]

# Calculate ROC-AUC for the single binary label
auc_score = roc_auc_score(y, y_pred_probs)

auc_scoret = roc_auc_score(yt, yt_pred_probs)

print(f"TRAINING ROC-AUC Score: {auc_score:.4f}, Test ROC-AUC Score: {auc_scoret:.4f}")


Training Data 0.894, Test Data Score: 0.928
TRAINING ROC-AUC Score: 0.9793, Test ROC-AUC Score: 0.9627


In [10]:
training_flagged = df[df['flag']==1]
Xflag = X[training_flagged.index]

In [11]:
target_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
models = []

for col in target_cols:
    print(col)
    y = training_flagged[col]
    yt = vtest[col]
    model = LogisticRegression(
                C = 0.5
                , solver='sag'
                , class_weight='balanced'
                , n_jobs = -1
            )
    if col == 'toxic':  # almost all flagged are toxic, so using the full dataset to train this
        model.fit(Xtrain,train[col])
    else:
        model.fit(Xflag,y)
    print('Training Data {:.3f}, Test Data Score: {:.3f}'.format(model.score(Xflag,y),model.score(Xt,yt)))
    models.append(model)

    y_pred_probs = model.predict_proba(Xflag)[:, 1]
    yt_pred_probs = model.predict_proba(Xt)[:, 1]

    auc_score = roc_auc_score(y, y_pred_probs)
    auc_scoret = roc_auc_score(yt, yt_pred_probs)
    # Calculate ROC-AUC for the single binary label
    print(f"TRAINING ROC-AUC Score: {auc_score:.4f}, Test ROC-AUC Score: {auc_scoret:.4f}")
    print()

toxic
Training Data 0.815, Test Data Score: 0.931
TRAINING ROC-AUC Score: 0.8931, Test ROC-AUC Score: 0.9622

severe_toxic


C:\Users\darin\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Training Data 0.922, Test Data Score: 0.985
TRAINING ROC-AUC Score: 0.9772, Test ROC-AUC Score: 0.9818

obscene
Training Data 0.900, Test Data Score: 0.961
TRAINING ROC-AUC Score: 0.9782, Test ROC-AUC Score: 0.9429

threat


C:\Users\darin\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Training Data 0.971, Test Data Score: 0.990
TRAINING ROC-AUC Score: 0.9893, Test ROC-AUC Score: 0.9842

insult
Training Data 0.882, Test Data Score: 0.962
TRAINING ROC-AUC Score: 0.9553, Test ROC-AUC Score: 0.9548

identity_hate


C:\Users\darin\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Training Data 0.962, Test Data Score: 0.979
TRAINING ROC-AUC Score: 0.9905, Test ROC-AUC Score: 0.9806



In [12]:
testing_gate = pd.DataFrame()
testing_gate['id'] = vtest['id']
testing_gate['tflag'] = gatekeeper.predict_proba(Xt)[:, 1]
cols = ['ttoxic', 'tsevere_toxic', 'tobscene', 'tthreat', 'tinsult', 'tidentity_hate']
for i, model in enumerate(models):
     testing_gate[cols[i]] = model.predict_proba(Xt)[:, 1]


ndf = vtest.copy().merge(testing_gate, on='id')
ndf.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,pos,stems,flag,tflag,ttoxic,tsevere_toxic,tobscene,tthreat,tinsult,tidentity_hate
0,0001ea8717f6de06,Thank you for understanding. I think very high...,0,0,0,0,0,0,NN PRP IN VBG JJ VBP RB RB IN PRP CC MD RB VB ...,thank understand i think veri highli not rever...,0,0.154833,0.137453,0.052705,0.159321,0.217008,0.281528,0.080173
1,000247e83dcc1211,:Dear god this site is horrible.,0,0,0,0,0,0,JJ NN DT NN VBZ JJ,dear god site is horribl,0,0.502930,0.519497,0.036414,0.112307,0.096032,0.183332,0.053904
2,0002f87b16116a7f,"""::: Somebody will invariably try to add Relig...",0,0,0,0,0,0,NN MD RB VB TO VB NN RB PRP VBP DT NN NNS VBP ...,somebodi invari tri add religion realli mean w...,0,0.275700,0.268616,0.045398,0.233167,0.015902,0.197926,0.133890
3,0003e1cccfd5a40a,""" \n\n It says it right there that it IS a typ...",0,0,0,0,0,0,PRP VBZ PRP RB EX IN PRP VBZ DT NN DT NN IN NN...,say right is type type institut is need case a...,0,0.163541,0.122696,0.023922,0.185517,0.010247,0.149974,0.100159
4,00059ace3e3e9a53,""" \n\n == Before adding a new product to the l...",0,0,0,0,0,0,IN VBG DT JJ NN TO DT NN VBP JJ PRP$ NN IN VBG...,ad new product list make sure relev ad new pro...,0,0.135986,0.143095,0.028530,0.176186,0.023091,0.111539,0.035937


In [13]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from sklearn.metrics import precision_recall_curve

def set_threshold(model = None, models = None, col = None, df=None, Xt=None):
    testing_gate = pd.DataFrame()
    testing_gate['id'] = df['id']
    testing_gate['tflag'] = gatekeeper.predict_proba(Xt)[:, 1]
    cols = ['ttoxic', 'tsevere_toxic', 'tobscene', 'tthreat', 'tinsult', 'tidentity_hate']
    if col != None:
        for i, model in enumerate(models):
             testing_gate[cols[i]] = model.predict_proba(Xt)[:, 1]
    
    ndf = df.copy().merge(testing_gate, on='id')

    cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    if col == None:
        col = 'flag'
    if model == None:
        model = models[cols.index(col)]

    if col == 'flag':
        # Get false positive rates, true positive rates, and thresholds
        fpr, tpr, thresholds = roc_curve(ndf[col], ndf['t'+col])
        j_scores = tpr - fpr
        best_idx = j_scores.argmax()
        best_threshold = thresholds[best_idx]
        return best_threshold
    else:
        precision, recall, thresholds = precision_recall_curve(ndf[col], ndf['t' + col])
        
        # Calculate F1 score for each threshold
        # (Adding 1e-10 prevents division by zero errors)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
        
        # Find the index of the highest F1 score
        best_idx = f1_scores.argmax()
        best_threshold = thresholds[best_idx]
        return best_threshold
    
def test_models(gatekeeper, models, gthresh, mthresh): 
    cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
   
    false_positives = 0
    false_negatives = 0
    true_positives = 0
    true_negatives = 0
    for i, row in ndf.iterrows():
        if row['flag'] == 0 and row['tflag'] < gthresh:
            true_negatives +=1
        elif row['flag'] == 0 and row['tflag'] >= gthresh:
            false_positives += 1
        elif row['flag'] == 1 and row['tflag'] < gthresh:
            false_negatives += 1
        else:
            checkfn = 0
            checkfp = 0
            for col in cols:
                if row[col] == 0 and row['t'+col] >= mthresh[cols.index(col)]:
                    checkfp += 1
                elif row[col] == 1 and row['t'+col] <  mthresh[cols.index(col)]:
                    checkfn += 1
            if checkfn >= 1:
                false_negatives += 1
    
            if checkfp >= 1:
                false_positives += 1
            if checkfn == 0 and checkfp == 0:
                true_positives += 1
  
    print('TP: {} TN: {} FP: {} FN: {}'.format(true_positives, true_negatives, false_positives, false_negatives))



gthresh = set_threshold(model=gatekeeper, df=vtest, Xt=Xt)
mthresh = [set_threshold(models=models, df=vtest, Xt=Xt, col=col) for col in ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']]
test_models(gatekeeper, models, gthresh, mthresh)

TP: 1912 TN: 50841 FP: 8199 FN: 3221


In [14]:
gthresh

0.36233940434064227

In [15]:
mthresh

[0.6105928808054819,
 0.7028615783493212,
 0.5312729948829492,
 0.844603616170427,
 0.5011497057331437,
 0.749991012252584]

In [16]:
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,pos,stems,flag,pred
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,NN WRB DT NNS VBN IN PRP$ JJ NN NN NN VBD VBN ...,explan whi edit made usernam hardcor metallica...,0,0.030489
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,NN PRP VBZ DT NN VBZ JJ RB VBN IN NNS NN CD NN...,daww match background colour im seemingli stuc...,0,0.019345
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,NN NN VBP RB RB VBG TO VB NN PRP$ RB IN DT NN ...,hey man im realli not tri edit war just guy is...,0,0.073244
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,RBR JJ JJ VBP DT JJ NNS IN NN NN VBD IN DT NN ...,more i cant make real suggest improv i wonder ...,0,0.002369
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,PRP VBP VBP PRP$ NN DT NN PRP VBP WP NN NNS IN,sir are hero chanc rememb page that,0,0.160274


In [17]:
vtest['pred'] = yt_pred_probs = gatekeeper.predict_proba(Xt)[:, 1]
vtest.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,pos,stems,flag,pred
0,0001ea8717f6de06,Thank you for understanding. I think very high...,0,0,0,0,0,0,NN PRP IN VBG JJ VBP RB RB IN PRP CC MD RB VB ...,thank understand i think veri highli not rever...,0,0.154833
1,000247e83dcc1211,:Dear god this site is horrible.,0,0,0,0,0,0,JJ NN DT NN VBZ JJ,dear god site is horribl,0,0.502930
2,0002f87b16116a7f,"""::: Somebody will invariably try to add Relig...",0,0,0,0,0,0,NN MD RB VB TO VB NN RB PRP VBP DT NN NNS VBP ...,somebodi invari tri add religion realli mean w...,0,0.275700
3,0003e1cccfd5a40a,""" \n\n It says it right there that it IS a typ...",0,0,0,0,0,0,PRP VBZ PRP RB EX IN PRP VBZ DT NN DT NN IN NN...,say right is type type institut is need case a...,0,0.163541
4,00059ace3e3e9a53,""" \n\n == Before adding a new product to the l...",0,0,0,0,0,0,IN VBG DT JJ NN TO DT NN VBP JJ PRP$ NN IN VBG...,ad new product list make sure relev ad new pro...,0,0.135986


In [18]:
df_full = pd.concat([df,vtest])
df_full.to_csv("data/processed_data.csv", index=False)
df_full.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,pos,stems,flag,pred
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,NN WRB DT NNS VBN IN PRP$ JJ NN NN NN VBD VBN ...,explan whi edit made usernam hardcor metallica...,0,0.030489
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,NN PRP VBZ DT NN VBZ JJ RB VBN IN NNS NN CD NN...,daww match background colour im seemingli stuc...,0,0.019345
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,NN NN VBP RB RB VBG TO VB NN PRP$ RB IN DT NN ...,hey man im realli not tri edit war just guy is...,0,0.073244
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,RBR JJ JJ VBP DT JJ NNS IN NN NN VBD IN DT NN ...,more i cant make real suggest improv i wonder ...,0,0.002369
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,PRP VBP VBP PRP$ NN DT NN PRP VBP WP NN NNS IN,sir are hero chanc rememb page that,0,0.160274


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0
